### EMA LSTM

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import random
from itertools import product

from sklearn.model_selection import StratifiedGroupKFold, LeaveOneGroupOut
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

# ── Reproducibility Setup ───────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# ── Configuration ───────────────────────────────────────────────────────────────
TIME_STEPS     = 2      # Use the past N days to predict the next day
N_SPLITS_INNER = 5      # Number of inner CV folds
PIDS           = []     # Need for anonymity
IGNORE         = ['PID', 'Date', 'day_of_week', 'hour_of_day', 'is_Weekend', 'withinrange']

# ── Data Loading Function ───────────────────────────────────────────────────────
def load_data(pid):
    df = pd.read_csv(f'EMA_0420/{pid}_ema.csv', parse_dates=['Date'])
    return df.drop_duplicates('Date').sort_values('Date')

# Identify available features
_sample = load_data(PIDS[0])
FEATURE_COLS = [c for c in _sample.columns if c not in IGNORE]
print(FEATURE_COLS)

# ── Sequence Creation: Skip Gaps in Dates ───────────────────────────────────────
def create_sequences_skip_gaps(df, time_steps):
    X, y, dates = [], [], []
    for i in range(len(df) - time_steps):
        start = df['Date'].iloc[i]
        end   = df['Date'].iloc[i + time_steps - 1]
        if end - start == pd.Timedelta(days=time_steps - 1):
            window = df.iloc[i : i + time_steps][FEATURE_COLS].values.astype(np.float32)
            target_idx = i + time_steps
            X.append(window)
            y.append(int(df['withinrange'].iloc[target_idx]))
            dates.append(df['Date'].iloc[target_idx])
    return np.array(X), np.array(y, dtype=int), np.array(dates)

# ── Load and Aggregate All Participants' Data ───────────────────────────────────
X_list, y_list, date_list, g_list = [], [], [], []
for pid in PIDS:
    df   = load_data(pid)
    Xi, yi, di = create_sequences_skip_gaps(df, TIME_STEPS)
    if len(yi):
        X_list.append(Xi)
        y_list.append(yi)
        date_list.append(di)
        g_list.append(np.full(len(yi), pid, dtype=int))

X_all    = np.concatenate(X_list, axis=0)
y_all    = np.concatenate(y_list, axis=0)
date_all = np.concatenate(date_list, axis=0)
groups   = np.concatenate(g_list, axis=0)

# ── LSTM Model Builder ──────────────────────────────────────────────────────────
def build_lstm_model(input_shape, units, dropout_rate, l2_reg):
    inp = Input(shape=input_shape)
    x   = LSTM(units, return_sequences=True, kernel_regularizer=l2(l2_reg))(inp)
    x   = Dropout(dropout_rate)(x)
    x   = LSTM(units, return_sequences=False, kernel_regularizer=l2(l2_reg))(x)
    x   = Dropout(dropout_rate)(x)
    x   = Dense(32, activation='relu', kernel_regularizer=l2(l2_reg))(x)
    x   = Dropout(dropout_rate)(x)
    out = Dense(1, activation='sigmoid', kernel_regularizer=l2(l2_reg))(x)
    model = Model(inp, out)
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

# ── Hyperparameter Search Space ─────────────────────────────────────────────────
param_grid = {
    'units':   [32, 64, 128],
    'dropout': [0.2, 0.3],
    'l2_reg':  [1e-3, 1e-4]
}
candidate_pool_size = 50
feature_sizes = [10, 20, 30, 40, 50]

# ── Cross-validation Setup ──────────────────────────────────────────────────────
outer_cv = LeaveOneGroupOut()
inner_cv = StratifiedGroupKFold(n_splits=N_SPLITS_INNER, shuffle=True, random_state=SEED)

# ── Evaluation Storage ──────────────────────────────────────────────────────────
all_true, all_prob, all_pred = [], [], []
all_pid, all_date = [], []
outer_ba_scores, outer_auc_scores = [], []

# ── Outer CV Loop ───────────────────────────────────────────────────────────────
for fold, (train_o, test_o) in enumerate(outer_cv.split(X_all, y_all, groups)):
    print(f"Fold {fold} - Test PIDs:", np.unique(groups[test_o]))
    X_tr_o, y_tr_o = X_all[train_o], y_all[train_o]
    X_te_o, y_te_o = X_all[test_o],  y_all[test_o]
    groups_tr_o    = groups[train_o]

    # Initial Feature Selection with Random Forest using the last day's features
    X_prev_o = X_tr_o[:, -1, :]
    rf_init  = RandomForestClassifier(n_estimators=100, random_state=SEED)
    rf_init.fit(X_prev_o, y_tr_o)
    imp_init = rf_init.feature_importances_
    candidate_idx = np.argsort(imp_init)[::-1][:candidate_pool_size]

    # ── Inner CV Loop for Hyperparameter & Feature Set Selection ────────────────
    best_inner_score = -np.inf
    best_params = {}
    best_feature_set = None

    for units, drop, l2_reg in product(param_grid['units'],
                                       param_grid['dropout'],
                                       param_grid['l2_reg']):
        for n_feats in feature_sizes:
            selected_idx = candidate_idx[:n_feats]
            inner_scores = []

            for tr_i, val_i in inner_cv.split(X_tr_o, y_tr_o, groups_tr_o):
                # Select features
                X_tr_sel = X_tr_o[tr_i][:, :, selected_idx]
                X_val_sel = X_tr_o[val_i][:, :, selected_idx]

                # Flatten and scale
                n_s, ts, nf = X_tr_sel.shape
                X_tr_flat = X_tr_sel.reshape(n_s, ts * nf)
                X_val_flat = X_val_sel.reshape(X_val_sel.shape[0], ts * nf)

                scaler = MinMaxScaler()
                X_tr_scaled = scaler.fit_transform(X_tr_flat)
                X_val_scaled = scaler.transform(X_val_flat)

                X_tr_i = X_tr_scaled.reshape(n_s, ts, nf)
                X_val_i = X_val_scaled.reshape(X_val_sel.shape[0], ts, nf)

                # Compute class weights for imbalance
                cw = compute_class_weight('balanced',
                                          classes=np.unique(y_tr_o[tr_i]),
                                          y=y_tr_o[tr_i])
                class_weights = {cls: w for cls, w in zip(np.unique(y_tr_o[tr_i]), cw)}

                # Train and evaluate model
                model = build_lstm_model((ts, nf), units, drop, l2_reg)
                es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
                model.fit(
                    X_tr_i, y_tr_o[tr_i],
                    validation_data=(X_val_i, y_tr_o[val_i]),
                    epochs=30, batch_size=32,
                    class_weight=class_weights,
                    callbacks=[es], verbose=0
                )

                prob_val = model.predict(X_val_i).ravel()
                pred_val = (prob_val > 0.5).astype(int)
                inner_scores.append(balanced_accuracy_score(y_tr_o[val_i], pred_val))

            mean_inner = np.mean(inner_scores)
            if mean_inner > best_inner_score:
                best_inner_score = mean_inner
                best_params = {
                    'units': units,
                    'dropout': drop,
                    'l2_reg': l2_reg,
                    'n_feats': n_feats
                }
                best_feature_set = selected_idx

    # ── Final Training & Testing with Best Parameters ───────────────────────────
    up = best_params
    mask_o = np.zeros_like(imp_init, dtype=bool)
    mask_o[best_feature_set] = True

    # Select and scale features
    X_tr_o_sel = X_tr_o[:, :, mask_o]
    X_te_o_sel = X_te_o[:, :, mask_o]

    n_tr, ts_o, nf_o = X_tr_o_sel.shape
    X_tr_flat = X_tr_o_sel.reshape(n_tr, ts_o * nf_o)
    X_te_flat = X_te_o_sel.reshape(X_te_o_sel.shape[0], ts_o * nf_o)

    scaler_o = MinMaxScaler()
    X_tr_scaled = scaler_o.fit_transform(X_tr_flat)
    X_te_scaled = scaler_o.transform(X_te_flat)

    X_tr_o_sel = X_tr_scaled.reshape(n_tr, ts_o, nf_o)
    X_te_o_sel = X_te_scaled.reshape(X_te_o_sel.shape[0], ts_o, nf_o)

    # Compute class weights again
    cw_o = compute_class_weight('balanced',
                                classes=np.unique(y_tr_o),
                                y=y_tr_o)
    class_weights_o = {cls: w for cls, w in zip(np.unique(y_tr_o), cw_o)}

    # Train final model on full outer training set
    final_model = build_lstm_model((ts_o, nf_o),
                                   up['units'], up['dropout'], up['l2_reg'])
    es_o = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    final_model.fit(
        X_tr_o_sel, y_tr_o,
        validation_split=0.2,
        epochs=30, batch_size=32,
        class_weight=class_weights_o,
        callbacks=[es_o], verbose=0
    )

    # Evaluate on the outer test set
    prob_test = final_model.predict(X_te_o_sel).ravel()
    pred_test = (prob_test > 0.5).astype(int)
    ba_test   = balanced_accuracy_score(y_te_o, pred_test)
    auc_test  = roc_auc_score(y_te_o, prob_test)

    print(f"Fold {fold} best_params={up} → BA={ba_test:.4f}, AUC={auc_test:.4f}")

    outer_ba_scores.append(ba_test)
    outer_auc_scores.append(auc_test)

    all_true .extend(y_te_o.tolist())
    all_prob .extend(prob_test.tolist())
    all_pred .extend(pred_test.tolist())
    all_pid  .extend(groups[test_o].tolist())
    all_date .extend(date_all[test_o].tolist())

# ── Save Results and Compute Overall Metrics ────────────────────────────────────
results_df = pd.DataFrame({
    'PID':        all_pid,
    'start_date': all_date,
    'true':       all_true,
    'pred':       all_pred,
    'prob':       all_prob
})
results_df.to_csv(f'results_ema_all_amplified_LOPO_{TIME_STEPS}.csv', index=False)

global_ba  = balanced_accuracy_score(all_true, all_pred)
global_auc = roc_auc_score(all_true, all_prob)
print(f"\nGlobal Balanced Accuracy: {global_ba:.4f}")
print(f"Global AUC:              {global_auc:.4f}")

### Fitbit LSTM

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import random
from itertools import product

from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

# ── Reproducibility ─────────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# ── Configuration ───────────────────────────────────────────────────────────────
TIME_STEPS      = 2
N_SPLITS_INNER  = 5    # Number of inner cross-validation folds
PIDS            = []    # Need for anonymity
IGNORE          = ["Date","withinrange"]  # Non-feature columns

# ── Data loading and feature identification ─────────────────────────────────────
def load_data(pid):
    df = pd.read_csv(f'daily_0418/{pid}_fitbit.csv', parse_dates=['Date'])
    return df.drop_duplicates('Date').sort_values('Date')

_sample      = load_data(PIDS[0])
FEATURE_COLS = [c for c in _sample.columns if c not in IGNORE]
print("Features:", FEATURE_COLS)

# ── Sequence construction with gap checking ─────────────────────────────────────
def create_sequences_skip_gaps(df, time_steps):
    X, y, dates = [], [], []
    for i in range(len(df) - time_steps):
        start = df['Date'].iloc[i]
        end   = df['Date'].iloc[i + time_steps - 1]
        if end - start == pd.Timedelta(days=time_steps - 1):
            window = df.iloc[i : i + time_steps][FEATURE_COLS].values.astype(np.float32)
            target_idx = i + time_steps
            X.append(window)
            y.append(int(df['withinrange'].iloc[target_idx]))
            dates.append(df['Date'].iloc[target_idx])
    return np.array(X), np.array(y, dtype=int), np.array(dates)

# ── Aggregate all participant data ──────────────────────────────────────────────
X_list, y_list, date_list, g_list = [], [], [], []
for pid in PIDS:
    df = load_data(pid)
    Xi, yi, di = create_sequences_skip_gaps(df, TIME_STEPS)
    if len(yi):
        X_list.append(Xi)
        y_list.append(yi)
        date_list.append(di)
        g_list.append(np.full(len(yi), pid, dtype=int))

X_all    = np.concatenate(X_list, axis=0)
y_all    = np.concatenate(y_list, axis=0)
date_all = np.concatenate(date_list, axis=0)
groups   = np.concatenate(g_list, axis=0)

# ── LSTM model definition ───────────────────────────────────────────────────────
def build_lstm_model(input_shape, units, dropout_rate, l2_reg):
    inp = Input(shape=input_shape)
    x   = LSTM(units, return_sequences=True, kernel_regularizer=l2(l2_reg))(inp)
    x   = Dropout(dropout_rate)(x)
    x   = LSTM(units, return_sequences=False, kernel_regularizer=l2(l2_reg))(x)
    x   = Dropout(dropout_rate)(x)
    x   = Dense(32, activation='relu', kernel_regularizer=l2(l2_reg))(x)
    x   = Dropout(dropout_rate)(x)
    out = Dense(1, activation='sigmoid', kernel_regularizer=l2(l2_reg))(x)
    model = Model(inp, out)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

# ── Hyperparameter and feature selection space ──────────────────────────────────
param_grid = {
    'units':   [32, 64, 128],
    'dropout': [0.2, 0.3],
    'l2_reg':  [1e-3, 1e-4]
}
candidate_pool_size = 50
feature_sizes = [20, 25, 30, 35, 40, 45, 50]

outer_cv = LeaveOneGroupOut()
inner_cv = StratifiedGroupKFold(n_splits=N_SPLITS_INNER, shuffle=True, random_state=SEED)

# ── Nested cross-validation with two-stage feature selection ────────────────────
all_true, all_prob, all_pred = [], [], []
all_pid, all_date = [], []
outer_ba_scores, outer_auc_scores = [], []

for fold, (train_o, test_o) in enumerate(outer_cv.split(X_all, y_all, groups)):
    print(f"Fold {fold} - Test PIDs:", np.unique(groups[test_o]))
    X_tr_o, y_tr_o = X_all[train_o], y_all[train_o]
    X_te_o, y_te_o = X_all[test_o],  y_all[test_o]
    groups_tr_o    = groups[train_o]

    # ── Stage 1: Initial feature screening via Random Forest ─────────────────────
    X_prev = X_tr_o[:, -1, :]  # Use the last day of each sequence
    rf_init = RandomForestClassifier(n_estimators=100, random_state=SEED)
    rf_init.fit(X_prev, y_tr_o)
    imp_init = rf_init.feature_importances_
    candidate_idx = np.argsort(imp_init)[::-1][:candidate_pool_size]

    best_inner_score = -np.inf
    best_params = None
    best_feature_set = None

    # ── Stage 2: Inner loop for hyperparameter tuning and refined feature selection ──
    for units, dropout, l2_reg in product(param_grid['units'], param_grid['dropout'], param_grid['l2_reg']):
        for n_feats in feature_sizes:
            selected_idx = candidate_idx[:n_feats]
            inner_scores = []
            for tr_i, val_i in inner_cv.split(X_tr_o, y_tr_o, groups_tr_o):
                # Slice features
                X_tr_sel = X_tr_o[tr_i][:, :, selected_idx]
                X_val_sel = X_tr_o[val_i][:, :, selected_idx]
                # Scale features
                n_s, ts, nf = X_tr_sel.shape
                X_tr_flat = X_tr_sel.reshape(n_s, ts * nf)
                X_val_flat = X_val_sel.reshape(X_val_sel.shape[0], ts * nf)
                scaler = MinMaxScaler()
                X_tr_scaled = scaler.fit_transform(X_tr_flat)
                X_val_scaled = scaler.transform(X_val_flat)
                X_tr_i = X_tr_scaled.reshape(n_s, ts, nf)
                X_val_i = X_val_scaled.reshape(X_val_sel.shape[0], ts, nf)
                # Compute class weights
                cw = compute_class_weight('balanced', classes=np.unique(y_tr_o[tr_i]), y=y_tr_o[tr_i])
                class_weights = {cls: w for cls, w in zip(np.unique(y_tr_o[tr_i]), cw)}
                # Train and evaluate
                model = build_lstm_model((ts, nf), units, dropout, l2_reg)
                es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
                model.fit(X_tr_i, y_tr_o[tr_i], validation_data=(X_val_i, y_tr_o[val_i]),
                          epochs=30, batch_size=32, class_weight=class_weights,
                          callbacks=[es], verbose=0)
                prob_val = model.predict(X_val_i).ravel()
                pred_val = (prob_val > 0.5).astype(int)
                inner_scores.append(balanced_accuracy_score(y_tr_o[val_i], pred_val))
            mean_inner = np.mean(inner_scores)
            if mean_inner > best_inner_score:
                best_inner_score = mean_inner
                best_params = {'units': units, 'dropout': dropout, 'l2_reg': l2_reg, 'n_feats': n_feats}
                best_feature_set = selected_idx

    # ── Final model retraining and testing on outer test fold ────────────────────
    mask_o = np.zeros_like(imp_init, dtype=bool)
    mask_o[best_feature_set] = True
    X_tr_sel_o = X_tr_o[:, :, mask_o]
    X_te_sel_o = X_te_o[:, :, mask_o]
    # Scale features
    n_tr, ts_o, nf_o = X_tr_sel_o.shape
    X_tr_flat = X_tr_sel_o.reshape(n_tr, ts_o * nf_o)
    X_te_flat = X_te_sel_o.reshape(X_te_sel_o.shape[0], ts_o * nf_o)
    scaler_o = MinMaxScaler()
    X_tr_scaled = scaler_o.fit_transform(X_tr_flat)
    X_te_scaled = scaler_o.transform(X_te_flat)
    X_tr_o_final = X_tr_scaled.reshape(n_tr, ts_o, nf_o)
    X_te_o_final = X_te_scaled.reshape(X_te_sel_o.shape[0], ts_o, nf_o)
    # Compute class weights
    cw_o = compute_class_weight('balanced', classes=np.unique(y_tr_o), y=y_tr_o)
    class_weights_o = {cls: w for cls, w in zip(np.unique(y_tr_o), cw_o)}
    # Train final model
    final_model = build_lstm_model((ts_o, nf_o),
                                   best_params['units'], best_params['dropout'], best_params['l2_reg'])
    es_o = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    final_model.fit(X_tr_o_final, y_tr_o, validation_split=0.2,
                    epochs=30, batch_size=32, class_weight=class_weights_o,
                    callbacks=[es_o], verbose=0)
    prob_test = final_model.predict(X_te_o_final).ravel()
    pred_test = (prob_test > 0.5).astype(int)
    ba_test = balanced_accuracy_score(y_te_o, pred_test)
    auc_test = roc_auc_score(y_te_o, prob_test)
    print(f"Fold {fold} params={best_params} → BA={ba_test:.4f}, AUC={auc_test:.4f}")
    outer_ba_scores.append(ba_test)
    outer_auc_scores.append(auc_test)
    all_true.extend(y_te_o.tolist())
    all_prob.extend(prob_test.tolist())
    all_pred.extend(pred_test.tolist())
    all_pid.extend(groups[test_o].tolist())
    all_date.extend(date_all[test_o].tolist())

# ── Save results and print global metrics ───────────────────────────────────────
results_df = pd.DataFrame({'PID': all_pid,
                           'start_date': all_date,
                           'true': all_true,
                           'pred': all_pred,
                           'prob': all_prob})
results_df.to_csv(f'results_fitbit_{TIME_STEPS}_LOPO.csv', index=False)

print(f"Global BA={balanced_accuracy_score(all_true, all_pred):.4f}, AUC={roc_auc_score(all_true, all_prob):.4f}")

### MEMS LSTM

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import random

from itertools import product
from sklearn.model_selection import StratifiedGroupKFold, LeaveOneGroupOut
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

# ── Reproducibility ─────────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# ── Settings ────────────────────────────────────────────────────────────────────
TIME_STEPS     = 2     
N_SPLITS_INNER = 5     # Number of folds in inner cross-validation
PIDS           = []    # Need for anonymity
FEATURE_COLS   = ["withinrange", "is_Weekend", "is_Night", "is_Morning", "is_Afternoon", "is_Evening"]

# ── Load and preprocess full RxCap dataset ──────────────────────────────────────
df_all = pd.read_csv('rx_cap_adherence_tz_aware_with_periods.csv', parse_dates=['Date'])

# Create time-windowed sequences for LSTM input
def create_sequences_rxcap(df, time_steps=7):
    X, y, target_dates, start_dates, groups = [], [], [], [], []
    for i in range(len(df) - time_steps):
        start = df['Date'].iloc[i]
        end   = df['Date'].iloc[i + time_steps - 1]
        if end - start == pd.Timedelta(days=time_steps - 1):
            window      = df.iloc[i : i + time_steps][FEATURE_COLS].values.astype(np.float32)
            label       = df['withinrange'].iloc[i + time_steps]
            target_date = df['Date'].iloc[i + time_steps]
            pid         = df['PID'].iloc[i]
            X.append(window)
            y.append(int(label))
            target_dates.append(target_date)
            start_dates.append(start)      # Record start date of the window
            groups.append(int(pid))
    return (
        np.array(X),
        np.array(y, dtype=int),
        np.array(target_dates),
        np.array(start_dates),
        np.array(groups, dtype=int)
    )

# ── Aggregate sequences from all PIDs ────────────────────────────────────────────
X_list, y_list, tar_list, start_list, g_list = [], [], [], [], []
for pid in PIDS:
    df_pid = (
        df_all[df_all['PID'] == pid]
        .drop_duplicates('Date')
        .sort_values('Date')
        .reset_index(drop=True)
    )
    Xi, yi, ti, si, gi = create_sequences_rxcap(df_pid, TIME_STEPS)
    if len(yi):
        X_list.append(Xi)
        y_list.append(yi)
        tar_list.append(ti)
        start_list.append(si)
        g_list.append(gi)

X_all      = np.concatenate(X_list, axis=0)
y_all      = np.concatenate(y_list, axis=0)
target_all = np.concatenate(tar_list, axis=0)   # Prediction date
start_all  = np.concatenate(start_list, axis=0) # Sequence start date
groups     = np.concatenate(g_list, axis=0)

# ── Cross-validation setup ───────────────────────────────────────────────────────
outer_cv = LeaveOneGroupOut()
inner_cv = StratifiedGroupKFold(n_splits=N_SPLITS_INNER, shuffle=True, random_state=SEED)

# ── LSTM model builder ──────────────────────────────────────────────────────────
def build_lstm_model(input_shape, units, dropout_rate, l2_reg):
    inp   = Input(shape=input_shape)
    x     = LSTM(units, return_sequences=True, kernel_regularizer=l2(l2_reg))(inp)
    x     = Dropout(dropout_rate)(x)
    x     = LSTM(units, return_sequences=False, kernel_regularizer=l2(l2_reg))(x)
    x     = Dropout(dropout_rate)(x)
    x     = Dense(32, activation='relu', kernel_regularizer=l2(l2_reg))(x)
    x     = Dropout(dropout_rate)(x)
    out   = Dense(1, activation='sigmoid', kernel_regularizer=l2(l2_reg))(x)
    model = Model(inp, out)
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(name='auc')]
    )
    return model

# ── Hyperparameter grid ─────────────────────────────────────────────────────────
param_grid = {
    'units':   [32, 64, 128],
    'dropout': [0.2, 0.3, 0.4],
    'l2_reg':  [1e-4, 1e-3],
}

# ── Nested Cross-Validation ─────────────────────────────────────────────────────
all_true, all_prob, all_pred = [], [], []
all_pid, all_start = [], []
outer_ba_scores, outer_auc_scores = [], []

for fold, (train_o, test_o) in enumerate(outer_cv.split(X_all, y_all, groups)):
    # Display test participants for the fold
    test_pids = np.unique(groups[test_o])
    print(f"Fold {fold} test PIDs:", test_pids)
    
    X_tr_o, y_tr_o = X_all[train_o], y_all[train_o]
    X_te_o, y_te_o = X_all[test_o],  y_all[test_o]
    groups_tr_o    = groups[train_o]

    # ── Inner loop for hyperparameter selection ─────────────────────────────────
    best_inner_score, best_params = -np.inf, None
    for units, drop, l2_reg in product(
            param_grid['units'],
            param_grid['dropout'],
            param_grid['l2_reg']
        ):
        inner_scores = []
        for tr_i, val_i in inner_cv.split(X_tr_o, y_tr_o, groups_tr_o):
            X_tr_i, y_tr_i = X_tr_o[tr_i], y_tr_o[tr_i]
            X_val_i, y_val_i = X_tr_o[val_i], y_tr_o[val_i]

            cw = compute_class_weight('balanced',
                                      classes=np.unique(y_tr_i),
                                      y=y_tr_i)
            class_weights = {cls: w for cls, w in zip(np.unique(y_tr_i), cw)}

            model = build_lstm_model(
                input_shape=X_tr_i.shape[1:],
                units=units,
                dropout_rate=drop,
                l2_reg=l2_reg
            )
            es = EarlyStopping(monitor='val_loss',
                               patience=5,
                               restore_best_weights=True)
            model.fit(
                X_tr_i, y_tr_i,
                validation_data=(X_val_i, y_val_i),
                epochs=30, batch_size=32,
                class_weight=class_weights,
                callbacks=[es],
                verbose=0
            )

            prob_val = model.predict(X_val_i).ravel()
            inner_scores.append(roc_auc_score(y_val_i, prob_val))

        mean_inner = np.mean(inner_scores)
        if mean_inner > best_inner_score:
            best_inner_score = mean_inner
            best_params      = dict(units=units, dropout=drop, l2_reg=l2_reg)

    # ── Retrain and evaluate on outer fold using best hyperparameters ───────────
    up = best_params
    cw_o = compute_class_weight('balanced',
                                classes=np.unique(y_tr_o),
                                y=y_tr_o)
    class_weights_o = {cls: w for cls, w in zip(np.unique(y_tr_o), cw_o)}

    final_model = build_lstm_model(
        input_shape=X_tr_o.shape[1:],
        units=up['units'],
        dropout_rate=up['dropout'],
        l2_reg=up['l2_reg']
    )
    es_o = EarlyStopping(monitor='val_loss',
                         patience=5,
                         restore_best_weights=True)
    final_model.fit(
        X_tr_o, y_tr_o,
        validation_split=0.2,
        epochs=30, batch_size=32,
        class_weight=class_weights_o,
        callbacks=[es_o],
        verbose=0
    )

    prob_test = final_model.predict(X_te_o).ravel()
    pred_test = (prob_test > 0.5).astype(int)
    ba_test   = balanced_accuracy_score(y_te_o, pred_test)
    auc_test  = roc_auc_score(y_te_o, prob_test)

    print(f"Fold {fold} params={up} → BA={ba_test:.4f}, AUC={auc_test:.4f}")

    outer_ba_scores.append(ba_test)
    outer_auc_scores.append(auc_test)

    # ── Accumulate predictions from the test set, using the window start date ───
    all_true.extend(y_te_o.tolist())
    all_prob.extend(prob_test.tolist())
    all_pred.extend(pred_test.tolist())
    all_pid.extend(groups[test_o].tolist())
    all_start.extend(start_all[test_o].tolist())  # Use start date of the window

# ── Save results to CSV ─────────────────────────────────────────────────────────
results_df = pd.DataFrame({
    'PID':        all_pid,
    'start_date': all_start,
    'true':       all_true,
    'pred':       all_pred,
    'prob':       all_prob
})
results_df.to_csv(f'results_mems_{TIME_STEPS}_LOPO.csv', index=False)

# ── Print global evaluation metrics ─────────────────────────────────────────────
global_ba  = balanced_accuracy_score(all_true, all_pred)
global_auc = roc_auc_score(all_true, all_prob)
print(f"\nGlobal Balanced Accuracy: {global_ba:.4f}")
print(f"Global AUC:              {global_auc:.4f}")